# **DSIO 2010 GROUP 4: Working Prototype**

---

The following code is part of the working prototype for the ***Clarity Analytics Center*** proposed in [DSIO2010_Group4_Proposal.pdf](https://drive.google.com/file/d/1VyIENVCsIUiZ7GTa-7D3oA20Qz7JNWen/view?usp=sharing). This notebook demonstrates the data processing core capablity using [python's sqlite3](https://docs.python.org/3/library/sqlite3.html). Raw data in the `bronze` layer is processed, transformed and moved through the medallion architecture up to the star schema in the `gold` layer. All sections are collapsible for better focus on desired code.<br><br>

<u>Contributing Members:</u> <br>
Hillary Ssemakula



---

### Imports and Utility Functions

In [1]:
# imports
import re
import sys
import time
import hashlib
import sqlite3
import pandas as pd
from pathlib import Path

! pip install tabulate
from tabulate import tabulate

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hilla\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# utility functions

def log(level, message):
    ts = time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime())
    print(f"{ts} {level.upper()}: {message}")

    if level.lower() == "error":
        raise Exception(message)
        sys.exit()


def get_md5_hash(*values):
    concat = '|'.join(str(v) for v in values)
    return hashlib.md5(concat.encode()).hexdigest()

def sql(conn, query):
    """Run a SELECT and return a DataFrame."""
    return pd.read_sql_query(query, conn)



def pretty_print_db(conn):
    """pretty print all tables in database with column and row count"""

    cursor = conn.cursor()
    cursor.execute(f'SELECT name FROM sqlite_master WHERE type="table" '
                    f'AND name != "sqlite_sequence"')
    tables = cursor.fetchall()

    table_data = []
    for table_name_tuple in tables:
        table_name = table_name_tuple[0]

        # get columns
        cursor.execute(f"PRAGMA table_info('{table_name}');")
        columns = cursor.fetchall()
        num_columns = len(columns)

        # get rows
        cursor.execute(f"SELECT COUNT(*) FROM '{table_name}';")
        num_rows = cursor.fetchone()[0]

        table_data.append([table_name, num_columns, num_rows])

    table_data.sort(key=lambda x: x[0])
    print("Tables in the database:")
    print(tabulate(table_data,
            headers=["Table Name", "Number of Columns", "Number of Rows"],
            tablefmt="fancy_grid"))


### **<br> <u>Database connection</u></br>**

In [3]:
db_file_name = 'clarity_analytics_center.db'
db_file_dir  = Path.cwd().parent.joinpath('storage/')
db_path = Path(db_file_dir).joinpath(db_file_name)
    
if not db_path.exists():
    log("error", f'The database file: "{db_path}" does not exist')


# connect to database file
try:
    conn = sqlite3.connect(db_path)
    print(f'Successfully connected to database: "{db_path}"')

    # check if database file is empty
    if not sql(conn, f'SELECT name FROM sqlite_master WHERE type="table" '
                    f'AND name != "sqlite_sequence"').empty:
        print()
        pretty_print_db(conn)

    else:
        log("error", "no tables found in the database")

except Exception as e:
    log("error", f'failed to connect to "{db_path}": {e}')


Successfully connected to database: "c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\clarity_analytics_center.db"

Tables in the database:
╒════════════════════════════════╤═════════════════════╤══════════════════╕
│ Table Name                     │   Number of Columns │   Number of Rows │
╞════════════════════════════════╪═════════════════════╪══════════════════╡
│ bronze_ice_budget              │                   7 │              300 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_enforcement_metrics │                   7 │               20 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_enforcement_pdfs    │                   6 │              125 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_operations          │                   9 │              400 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_

### **<br> <u>Transformation </u></br>**

### Silver

In [4]:
# operations
new_silver_operations_rows_df = sql(
    conn,
    """SELECT
           agent_id,
           supervisor_id,
           region,
           role,
           department,
           unit,
           detention_center_name,
           assignment_start_date,
           assignment_end_date
       FROM bronze_ice_operations""",
)

new_silver_operations_rows_df['operations_row_id'] = (
    new_silver_operations_rows_df['agent_id']
)
new_silver_operations_rows_df['is_active_assignment'] = (
    new_silver_operations_rows_df['assignment_end_date'].isnull().astype(int)
)

# correct HSI department value discrepancies:
new_silver_operations_rows_df.loc[
    new_silver_operations_rows_df['department'].str.contains('(HSI)', na=False, regex=False),
    'department'
] = 'Homeland Security Investigations (HSI)'

# drop incoming bronze rows that already exist in silver
silver_operations_agent_id_df = sql(conn, "SELECT agent_id FROM silver_operations")
new_silver_operations_rows_df = new_silver_operations_rows_df[
    ~new_silver_operations_rows_df['agent_id'].isin(
        silver_operations_agent_id_df['agent_id']
    )
]

if new_silver_operations_rows_df.empty:
    log("warn", "no new rows to load into silver_operations")
else:
    new_silver_operations_rows_df.to_sql(
        "silver_operations", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_silver_operations_rows_df)}] into silver_operations")

2026-08-13T22:43:25 WARN: no new rows to load into silver_operations


In [5]:
# budget
new_silver_budget_rows_df = sql(
    conn,
    """SELECT
           fiscal_year,
           department_code,
           agency_name,
           net_operating_cost,
           budget_authority,
           budget_deficit_contribution,
           source
       FROM bronze_ice_budget
       WHERE agency_name = 'Immigration and Customs Enforcement'""",
)

# generate budget_row_id using md5 hash as all columns are needed for row uniqueness
new_silver_budget_rows_df['budget_row_id'] = new_silver_budget_rows_df.apply(
    lambda row: get_md5_hash(*row.values), axis=1
)

# drop incoming bronze rows that already exist in silver
silver_budget_row_id_df = sql(conn, "SELECT budget_row_id FROM silver_budget")
new_silver_budget_rows_df = new_silver_budget_rows_df[
    ~new_silver_budget_rows_df['budget_row_id'].isin(
        silver_budget_row_id_df['budget_row_id']
    )
]

if new_silver_budget_rows_df.empty:
    log("warn", "no new rows to load into silver_budget")
else:
    new_silver_budget_rows_df.to_sql(
        "silver_budget", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_silver_budget_rows_df)}] into silver_budget")

2026-08-13T22:43:25 INFO: loaded [61] into silver_budget


In [6]:
# treasury_reconciliation
new_silver_treasury_reconciliation_rows_df = sql(
    conn,
    """SELECT
           record_date,
           stmt_fiscal_year,
           restmt_flag,
           account_desc,
           component_desc,
           line_item_desc,
           position_bil_amt,
           src_line_nbr,
           record_fiscal_year,
           record_fiscal_quarter,
           record_calendar_year,
           record_calendar_quarter,
           record_calendar_month,
           record_calendar_day
       FROM bronze_treasury_reconciliation""",
)

# rename columns to match silver column names
new_silver_treasury_reconciliation_rows_df = (
    new_silver_treasury_reconciliation_rows_df.rename(columns={
        'stmt_fiscal_year': 'statement_fiscal_year',
        'restmt_flag': 'restatement_flag',
        'account_desc': 'account_description',
        'component_desc': 'component_description',
        'line_item_desc': 'line_item_description',
        'position_bil_amt': 'position_billion_amount',
        'src_line_nbr': 'source_line_number',
    })
)

# convert column data types to match silver data types
int_columns = [
    'statement_fiscal_year',
    'source_line_number',
    'record_fiscal_year',
    'record_fiscal_quarter',
    'record_calendar_year',
    'record_calendar_quarter',
    'record_calendar_month',
    'record_calendar_day',
]
for column in int_columns:
    new_silver_treasury_reconciliation_rows_df[column] = (
        new_silver_treasury_reconciliation_rows_df[column].astype(int)
    )

new_silver_treasury_reconciliation_rows_df['position_billion_amount'] = (
    new_silver_treasury_reconciliation_rows_df['position_billion_amount'].astype(float)
)

# correct "Pensions and accrued benefits" line_item_description value discrepancies:
new_silver_treasury_reconciliation_rows_df.loc[
    new_silver_treasury_reconciliation_rows_df['line_item_description'].str.contains('Pension', na=False, regex=False),
    'line_item_description'
] = 'Pension and accrued benefits'

# remove erroneous extra colons and footnotes appended to values for
# account_description, component_description and line_item_description
malformed_columns = [
    'account_description',
    'component_description',
    'line_item_description'
]
for column in malformed_columns:
    new_silver_treasury_reconciliation_rows_df.loc[
        new_silver_treasury_reconciliation_rows_df[column].str.endswith(':'),
        column
    ] = new_silver_treasury_reconciliation_rows_df[column].str[:-1]

    new_silver_treasury_reconciliation_rows_df.loc[
        new_silver_treasury_reconciliation_rows_df[column] == 'Budget deficit1',
        column
    ] = 'Budget deficit'



# generate reconciliation_row_id using md5 hash of unique key(record_date, source_line_number, restatement_flag)
new_silver_treasury_reconciliation_rows_df['treasury_reconciliation_row_id'] = (
    new_silver_treasury_reconciliation_rows_df.apply(
        lambda row: get_md5_hash(
            row['record_date'], row['source_line_number'], row['restatement_flag']
        ),
        axis=1,
    )
)

# drop incoming bronze rows that already exist in silver
silver_treasury_reconciliation_row_id_df = sql(
    conn,
    "SELECT treasury_reconciliation_row_id FROM silver_treasury_reconciliation",
)
new_silver_treasury_reconciliation_rows_df = new_silver_treasury_reconciliation_rows_df[
    ~new_silver_treasury_reconciliation_rows_df['treasury_reconciliation_row_id'].isin(
        silver_treasury_reconciliation_row_id_df['treasury_reconciliation_row_id']
    )
]

if new_silver_treasury_reconciliation_rows_df.empty:
    log("warn", "no new rows to load into silver_treasury_reconciliation")
else:
    new_silver_treasury_reconciliation_rows_df.to_sql(
        "silver_treasury_reconciliation", conn, if_exists="append", index=False
    )
    log(
        "info",
        f"loaded [{len(new_silver_treasury_reconciliation_rows_df)}] "
        f"into silver_treasury_reconciliation",
    )

2026-08-13T22:43:25 WARN: no new rows to load into silver_treasury_reconciliation


In [7]:
# enforcement
enforcement_pdfs_df = sql(conn, """SELECT
                                      extraction_timestamp,
                                      source_file,
                                      page_number,
                                      content
                                  FROM bronze_ice_enforcement_pdfs""")

enforcement_metrics_df = sql(conn, """SELECT
                                          source_file,
                                          page_number,
                                          metric_name,
                                          metric_value,
                                          metric_comment
                                      FROM bronze_ice_enforcement_metrics""")

# match source_file in pdfs to metrics
enforcement_pdfs_df['source_file'] = (
    enforcement_pdfs_df['source_file'].str.rsplit('/', n=1).str[-1]
)

# extract report_year for all pdfs
enforcement_pdfs_df['report_year'] = (
    enforcement_pdfs_df['source_file'].str.extract(r'FY_(\d{4})_').astype(int)
)

# merge pdfs with enforcement metrics using left join
new_silver_enforcement_rows_df = enforcement_pdfs_df.merge(
    enforcement_metrics_df,
    how='left',
    left_on=['source_file', 'page_number'],
    right_on=['source_file', 'page_number'],
)

# generate enforcement_row_id using md5 hash of unique key(source_file, page_number, metric_value)
new_silver_enforcement_rows_df['enforcement_row_id'] = (
    new_silver_enforcement_rows_df.apply(
        lambda row: get_md5_hash(
            row['source_file'], row['page_number'], row['metric_value']
        ),
        axis=1,
    )
)

# drop incoming bronze rows that already exist in silver
silver_enforcement_row_id_df = sql(
    conn, "SELECT enforcement_row_id FROM silver_enforcement"
)
new_silver_enforcement_rows_df = new_silver_enforcement_rows_df[
    ~new_silver_enforcement_rows_df['enforcement_row_id'].isin(
        silver_enforcement_row_id_df['enforcement_row_id']
    )
]

if new_silver_enforcement_rows_df.empty:
    log("warn", "no new rows to load into silver_enforcement")
else:
    new_silver_enforcement_rows_df.to_sql(
        "silver_enforcement", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_silver_enforcement_rows_df)}] into silver_enforcement")

2026-08-13T22:43:25 WARN: no new rows to load into silver_enforcement


### Gold

In [8]:
# DIMENSIONS

In [9]:
# gold_dim_date
# the conformed date dimension. every fact joins here, which is what lets
# budget, enforcement, treasury and staffing reach each other.
date_span_df = sql(
    conn,
    """SELECT
           MIN(d) AS min_date,
           MAX(d) AS max_date
       FROM (
           SELECT MIN(assignment_start_date) AS d FROM silver_operations
           UNION ALL SELECT MAX(assignment_end_date) FROM silver_operations
           UNION ALL SELECT MIN(record_date) FROM silver_treasury_reconciliation
           UNION ALL SELECT MAX(record_date) FROM silver_treasury_reconciliation
           UNION ALL SELECT MIN(statement_fiscal_year) || '-10-01' FROM silver_treasury_reconciliation
           UNION ALL SELECT MAX(statement_fiscal_year) || '-09-30' FROM silver_treasury_reconciliation
           UNION ALL SELECT MIN(fiscal_year) || '-10-01' FROM silver_budget
           UNION ALL SELECT MAX(fiscal_year) || '-09-30' FROM silver_budget
       )
       WHERE d IS NOT NULL""",
)

span_start = pd.Timestamp(date_span_df['min_date'].iloc[0])
span_end = pd.Timestamp(date_span_df['max_date'].iloc[0])
calendar_dates = pd.date_range(
    start=pd.Timestamp(year=span_start.year - 1, month=10, day=1),
    end=pd.Timestamp(year=span_end.year + 1, month=9, day=30),
    freq='D',
)
month_end_dates = calendar_dates + pd.offsets.MonthEnd(0)

new_gold_dim_date_rows_df = pd.DataFrame({
    'date_key': calendar_dates.strftime('%Y%m%d').astype(int),
    'full_date': calendar_dates.strftime('%Y-%m-%d'),
    'calendar_year': calendar_dates.year,
    'calendar_quarter': calendar_dates.quarter,
    'calendar_month': calendar_dates.month,
    'calendar_day': calendar_dates.day,
    'month_name': calendar_dates.strftime('%b'),
    'day_name': calendar_dates.strftime('%a'),

    # US federal fiscal year starts 1 October
    'fiscal_year': calendar_dates.year + (calendar_dates.month >= 10).astype(int),
    'fiscal_quarter': ((calendar_dates.month + 2) % 12) // 3 + 1,
    'month_end_date': month_end_dates.strftime('%Y-%m-%d'),
    'is_month_end': (calendar_dates == month_end_dates).astype(int),
    'is_fiscal_year_end': (calendar_dates.strftime('%m-%d') == '09-30').astype(int),
    'is_weekend': (calendar_dates.dayofweek >= 5).astype(int),
})

new_gold_dim_date_rows_df['fiscal_year_label'] = (
    'FY' + new_gold_dim_date_rows_df['fiscal_year'].astype(str)
)

new_gold_dim_date_rows_df = new_gold_dim_date_rows_df[
    ['date_key', 'full_date', 'calendar_year', 'calendar_quarter',
     'calendar_month', 'calendar_day', 'month_name', 'day_name', 'fiscal_year',
     'fiscal_quarter', 'fiscal_year_label', 'month_end_date', 'is_month_end',
     'is_fiscal_year_end', 'is_weekend']
]

# drop incoming rows that already exist in gold
gold_dim_date_key_df = sql(conn, "SELECT date_key FROM gold_dim_date")

new_gold_dim_date_rows_df = new_gold_dim_date_rows_df[
    ~new_gold_dim_date_rows_df['date_key'].isin(gold_dim_date_key_df['date_key'])
]

if new_gold_dim_date_rows_df.empty:
    log("warn", "no new rows to load into gold_dim_date")
else:
    new_gold_dim_date_rows_df.to_sql(
        "gold_dim_date", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_dim_date_rows_df)}] into gold_dim_date")

2026-08-13T22:43:25 WARN: no new rows to load into gold_dim_date


In [10]:
# gold_dim_org_unit
new_gold_dim_org_unit_rows_df = sql(
    conn,
    """SELECT DISTINCT
           department AS department_name,
           unit       AS unit_name
       FROM silver_operations""",
)

new_gold_dim_org_unit_rows_df['org_unit_label'] = (
    new_gold_dim_org_unit_rows_df['department_name']
    + ' — ' + new_gold_dim_org_unit_rows_df['unit_name']
)

# generate org_unit_key using md5 hash of unique key(department_name, unit_name).
new_gold_dim_org_unit_rows_df['org_unit_key'] = (
    new_gold_dim_org_unit_rows_df.apply(
        lambda row: get_md5_hash(row['department_name'], row['unit_name']), axis=1
    )
)

# drop incoming rows that already exist in gold
gold_dim_org_unit_key_df = sql(conn, "SELECT org_unit_key FROM gold_dim_org_unit")
new_gold_dim_org_unit_rows_df = new_gold_dim_org_unit_rows_df[
    ~new_gold_dim_org_unit_rows_df['org_unit_key'].isin(
        gold_dim_org_unit_key_df['org_unit_key']
    )
]

if new_gold_dim_org_unit_rows_df.empty:
    log("warn", "no new rows to load into gold_dim_org_unit")
else:
    new_gold_dim_org_unit_rows_df.to_sql(
        "gold_dim_org_unit", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_dim_org_unit_rows_df)}] into gold_dim_org_unit")

2026-08-13T22:43:25 WARN: no new rows to load into gold_dim_org_unit


In [11]:
# gold_dim_role
new_gold_dim_role_rows_df = sql(
    conn, "SELECT DISTINCT role AS role_name FROM silver_operations"
)

ROLE_SENIORITY = {
    'Analyst': 1,
    'Agent': 2,
    'Senior Agent': 3,
    'Supervisor': 4,
    'Deputy Director': 5,
}
LEADERSHIP_ROLES = ['Supervisor', 'Deputy Director']

new_gold_dim_role_rows_df['seniority_rank'] = (
    new_gold_dim_role_rows_df['role_name'].map(ROLE_SENIORITY).fillna(0).astype(int)
)
new_gold_dim_role_rows_df['is_leadership'] = (
    new_gold_dim_role_rows_df['role_name'].isin(LEADERSHIP_ROLES).astype(int)
)

# a role outside the known set would silently rank 0, so stop instead
unranked_roles = new_gold_dim_role_rows_df.loc[
    new_gold_dim_role_rows_df['seniority_rank'] == 0, 'role_name'
].tolist()
if unranked_roles:
    log("error", f"role(s) missing from ROLE_SENIORITY: {unranked_roles}")

# generate role_key using md5 hash of unique key(role_name)
new_gold_dim_role_rows_df['role_key'] = new_gold_dim_role_rows_df.apply(
    lambda row: get_md5_hash(row['role_name']), axis=1
)

# drop incoming rows that already exist in gold
gold_dim_role_key_df = sql(conn, "SELECT role_key FROM gold_dim_role")
new_gold_dim_role_rows_df = new_gold_dim_role_rows_df[
    ~new_gold_dim_role_rows_df['role_key'].isin(gold_dim_role_key_df['role_key'])
]

if new_gold_dim_role_rows_df.empty:
    log("warn", "no new rows to load into gold_dim_role")
else:
    new_gold_dim_role_rows_df.to_sql(
        "gold_dim_role", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_dim_role_rows_df)}] into gold_dim_role")

2026-08-13T22:43:25 WARN: no new rows to load into gold_dim_role


In [12]:
# gold_dim_facility
new_gold_dim_facility_rows_df = sql(
    conn, "SELECT DISTINCT detention_center_name FROM silver_operations"
)

# split "Eloy Detention Center (AZ)" into facility name and state code
facility_parts_df = new_gold_dim_facility_rows_df['detention_center_name'].str.extract(
    r'^(?P<facility_name>.+?)\s*\((?P<state_code>[A-Z]{2})\)$'
)
new_gold_dim_facility_rows_df = pd.concat(
    [new_gold_dim_facility_rows_df, facility_parts_df], axis=1
)

unparsed_facilities = new_gold_dim_facility_rows_df.loc[
    new_gold_dim_facility_rows_df['facility_name'].isnull(), 'detention_center_name'
].tolist()
if unparsed_facilities:
    log("error", f"detention_center_name did not match 'Name (ST)': {unparsed_facilities}")

new_gold_dim_facility_rows_df['facility_type'] = 'Other'
new_gold_dim_facility_rows_df.loc[
    new_gold_dim_facility_rows_df['facility_name'].str.contains(
        'Processing Center', na=False),
    'facility_type'
] = 'ICE Processing Center'
new_gold_dim_facility_rows_df.loc[
    new_gold_dim_facility_rows_df['facility_name'].str.contains(
        'Detention Center', na=False),
    'facility_type'
] = 'Detention Center'

new_gold_dim_facility_rows_df = new_gold_dim_facility_rows_df.rename(
    columns={'detention_center_name': 'facility_label'}
)

# generate facility_key using md5 hash of unique key(facility_name, state_code)
new_gold_dim_facility_rows_df['facility_key'] = (
    new_gold_dim_facility_rows_df.apply(
        lambda row: get_md5_hash(row['facility_name'], row['state_code']), axis=1
    )
)

new_gold_dim_facility_rows_df = new_gold_dim_facility_rows_df[
    ['facility_key', 'facility_name', 'state_code', 'facility_type', 'facility_label']
]

# drop incoming rows that already exist in gold
gold_dim_facility_key_df = sql(conn, "SELECT facility_key FROM gold_dim_facility")
new_gold_dim_facility_rows_df = new_gold_dim_facility_rows_df[
    ~new_gold_dim_facility_rows_df['facility_key'].isin(
        gold_dim_facility_key_df['facility_key']
    )
]

if new_gold_dim_facility_rows_df.empty:
    log("warn", "no new rows to load into gold_dim_facility")
else:
    new_gold_dim_facility_rows_df.to_sql(
        "gold_dim_facility", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_dim_facility_rows_df)}] into gold_dim_facility")

2026-08-13T22:43:25 WARN: no new rows to load into gold_dim_facility


In [13]:
# gold_dim_treasury_line
new_gold_dim_treasury_line_rows_df = sql(
    conn,
    """SELECT DISTINCT
           account_description,
           component_description,
           line_item_description
       FROM silver_treasury_reconciliation""",
)

# which side of the accrual-to-cash walk each line sits on
RECONCILIATION_SIDES = {
    'Net operating cost': 'Net operating cost',
    'Budget deficit': 'Budget deficit',
    'Unified budget deficit': 'Budget deficit',
    'Adjustments to beginning net position': 'Net position adjustment',
    'Unmatched transactions and balances': 'Unmatched',
}

new_gold_dim_treasury_line_rows_df['reconciliation_side'] = (
    new_gold_dim_treasury_line_rows_df['account_description']
    .map(RECONCILIATION_SIDES).fillna('Other')
)
new_gold_dim_treasury_line_rows_df.loc[
    new_gold_dim_treasury_line_rows_df['account_description'].str.startswith(
        'Components of net operating cost'),
    'reconciliation_side'
] = 'Accrual-only (in NOC, not deficit)'
new_gold_dim_treasury_line_rows_df.loc[
    new_gold_dim_treasury_line_rows_df['account_description'].str.startswith(
        'Components of the budget deficit'),
    'reconciliation_side'
] = 'Cash-only (in deficit, not NOC)'

# subtotal and grand total rows share the same source column as detail rows.
# summing without filtering to DETAIL roughly quadruples the answer.
TOTAL_LINE_ITEMS = ['Net operating cost', 'Budget deficit', 'Unified budget deficit']

new_gold_dim_treasury_line_rows_df['line_level'] = 'DETAIL'
new_gold_dim_treasury_line_rows_df.loc[
    new_gold_dim_treasury_line_rows_df['line_item_description'].isin(TOTAL_LINE_ITEMS),
    'line_level'
] = 'TOTAL'
new_gold_dim_treasury_line_rows_df.loc[
    new_gold_dim_treasury_line_rows_df['line_item_description'].str.startswith('Subtotal'),
    'line_level'
] = 'SUBTOTAL'

new_gold_dim_treasury_line_rows_df['is_additive'] = (
    (new_gold_dim_treasury_line_rows_df['line_level'] == 'DETAIL').astype(int)
)

# generate treasury_line_key using md5 hash of unique key(component_description, line_item_description).
new_gold_dim_treasury_line_rows_df['treasury_line_key'] = (
    new_gold_dim_treasury_line_rows_df.apply(
        lambda row: get_md5_hash(
            row['component_description'], row['line_item_description']
        ),
        axis=1,
    )
)

new_gold_dim_treasury_line_rows_df = (
    new_gold_dim_treasury_line_rows_df
    .drop_duplicates(subset=['treasury_line_key'])[
        ['treasury_line_key', 'account_description', 'component_description',
         'line_item_description', 'reconciliation_side', 'line_level', 'is_additive']
    ]
)

# drop incoming rows that already exist in gold
gold_dim_treasury_line_key_df = sql(
    conn, "SELECT treasury_line_key FROM gold_dim_treasury_line"
)
new_gold_dim_treasury_line_rows_df = new_gold_dim_treasury_line_rows_df[
    ~new_gold_dim_treasury_line_rows_df['treasury_line_key'].isin(
        gold_dim_treasury_line_key_df['treasury_line_key']
    )
]

if new_gold_dim_treasury_line_rows_df.empty:
    log("warn", "no new rows to load into gold_dim_treasury_line")
else:
    new_gold_dim_treasury_line_rows_df.to_sql(
        "gold_dim_treasury_line", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_dim_treasury_line_rows_df)}] "
                f"into gold_dim_treasury_line")

2026-08-13T22:43:25 WARN: no new rows to load into gold_dim_treasury_line


In [14]:
# FACTS

In [15]:
# gold_fact_budget
new_gold_fact_budget_rows_df = sql(
    conn,
    """SELECT
           fiscal_year,
           department_code,
           net_operating_cost,
           budget_authority,
           budget_deficit_contribution
       FROM silver_budget""",
)


new_gold_fact_budget_rows_df['accrual_to_cash_gap'] = (
    new_gold_fact_budget_rows_df['net_operating_cost']
    - new_gold_fact_budget_rows_df['budget_deficit_contribution']
)

# budget is reported at fiscal year end, 30 September
new_gold_fact_budget_rows_df['date_key'] = pd.to_datetime(
    new_gold_fact_budget_rows_df['fiscal_year'].astype(str) + '-09-30'
).dt.strftime('%Y%m%d').astype('Int64')

# generate budget_key using md5 hash as all columns are needed for row uniqueness
new_gold_fact_budget_rows_df['budget_key'] = new_gold_fact_budget_rows_df.apply(
    lambda row: get_md5_hash(
        row['fiscal_year'], row['department_code'], row['net_operating_cost'],
        row['budget_authority'], row['budget_deficit_contribution']
    ),
    axis=1,
)

new_gold_fact_budget_rows_df = new_gold_fact_budget_rows_df[
    ['budget_key', 'date_key', 'fiscal_year', 'department_code',
     'net_operating_cost', 'budget_authority', 'budget_deficit_contribution',
     'accrual_to_cash_gap']
]

# stop rather than orphan a fact row against a missing dimension member
gold_dim_date_key_df = sql(conn, "SELECT date_key FROM gold_dim_date")
orphan_count = (~new_gold_fact_budget_rows_df['date_key'].isin(
    gold_dim_date_key_df['date_key'])).sum()
if orphan_count:
    log("error", f"[{orphan_count}] gold_fact_budget rows have a date_key with no "
                 f"matching gold_dim_date member")

# drop incoming rows that already exist in gold
gold_fact_budget_key_df = sql(conn, "SELECT budget_key FROM gold_fact_budget")
new_gold_fact_budget_rows_df = new_gold_fact_budget_rows_df[
    ~new_gold_fact_budget_rows_df['budget_key'].isin(
        gold_fact_budget_key_df['budget_key']
    )
]

if new_gold_fact_budget_rows_df.empty:
    log("warn", "no new rows to load into gold_fact_budget")
else:
    new_gold_fact_budget_rows_df.to_sql(
        "gold_fact_budget", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_fact_budget_rows_df)}] into gold_fact_budget")

2026-08-13T22:43:25 INFO: loaded [61] into gold_fact_budget


In [16]:
# gold_fact_treasury
new_gold_fact_treasury_rows_df = sql(
    conn,
    """SELECT
           record_date,
           source_line_number,
           statement_fiscal_year,
           record_fiscal_year,
           restatement_flag,
           component_description,
           line_item_description,
           position_billion_amount
       FROM silver_treasury_reconciliation""",
)


new_gold_fact_treasury_rows_df['is_restated'] = (
    (new_gold_fact_treasury_rows_df['restatement_flag'] == 'Y').astype(int)
)

new_gold_fact_treasury_rows_df['date_key'] = pd.to_datetime(
    new_gold_fact_treasury_rows_df['record_date']
).dt.strftime('%Y%m%d').astype('Int64')

# resolve the dimension key with the same hash used to build the dimension
new_gold_fact_treasury_rows_df['treasury_line_key'] = (
    new_gold_fact_treasury_rows_df.apply(
        lambda row: get_md5_hash(
            row['component_description'], row['line_item_description']
        ),
        axis=1,
    )
)

# generate treasury_key using md5 hash of unique key(record_date,
# source_line_number, restatement_flag)
new_gold_fact_treasury_rows_df['treasury_key'] = (
    new_gold_fact_treasury_rows_df.apply(
        lambda row: get_md5_hash(
            row['record_date'], row['source_line_number'], row['restatement_flag']
        ),
        axis=1,
    )
)

new_gold_fact_treasury_rows_df = new_gold_fact_treasury_rows_df[
    ['treasury_key', 'treasury_line_key', 'date_key', 'statement_fiscal_year',
     'record_fiscal_year', 'restatement_flag', 'is_restated', 'source_line_number',
     'position_billion_amount']
]

# stop rather than orphan a fact row against a missing dimension member
gold_dim_treasury_line_key_df = sql(
    conn, "SELECT treasury_line_key FROM gold_dim_treasury_line"
)
orphan_count = (~new_gold_fact_treasury_rows_df['treasury_line_key'].isin(
    gold_dim_treasury_line_key_df['treasury_line_key'])).sum()
if orphan_count:
    log("error", f"[{orphan_count}] gold_fact_treasury rows have a treasury_line_key "
                 f"with no matching gold_dim_treasury_line member")

gold_dim_date_key_df = sql(conn, "SELECT date_key FROM gold_dim_date")
orphan_count = (~new_gold_fact_treasury_rows_df['date_key'].isin(
    gold_dim_date_key_df['date_key'])).sum()
if orphan_count:
    log("error", f"[{orphan_count}] gold_fact_treasury rows have a date_key with no "
                 f"matching gold_dim_date member")

# drop incoming rows that already exist in gold
gold_fact_treasury_key_df = sql(conn, "SELECT treasury_key FROM gold_fact_treasury")
new_gold_fact_treasury_rows_df = new_gold_fact_treasury_rows_df[
    ~new_gold_fact_treasury_rows_df['treasury_key'].isin(
        gold_fact_treasury_key_df['treasury_key']
    )
]

if new_gold_fact_treasury_rows_df.empty:
    log("warn", "no new rows to load into gold_fact_treasury")
else:
    new_gold_fact_treasury_rows_df.to_sql(
        "gold_fact_treasury", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_fact_treasury_rows_df)}] into gold_fact_treasury")

2026-08-13T22:43:25 WARN: no new rows to load into gold_fact_treasury

In [17]:
# gold_fact_enforcement: one row per reported figure (fiscal year x metric)
new_gold_fact_enforcement_rows_df = sql(
    conn,
    """SELECT
           report_year AS fiscal_year,
           metric_name,
           metric_value,
           metric_comment,
           page_number AS source_page
       FROM silver_enforcement
       WHERE metric_name IS NOT NULL
         AND metric_value IS NOT NULL""",
)

# enforcement figures are reported at fiscal year end, 30 September
new_gold_fact_enforcement_rows_df['date_key'] = pd.to_datetime(
    new_gold_fact_enforcement_rows_df['fiscal_year'].astype(str) + '-09-30'
).dt.strftime('%Y%m%d').astype('Int64')

# generate enforcement_key using md5 hash of unique key(fiscal_year, metric_name)
new_gold_fact_enforcement_rows_df['enforcement_key'] = (
    new_gold_fact_enforcement_rows_df.apply(
        lambda row: get_md5_hash(row['fiscal_year'], row['metric_name']), axis=1
    )
)

new_gold_fact_enforcement_rows_df = new_gold_fact_enforcement_rows_df[
    ['enforcement_key', 'date_key', 'fiscal_year', 'metric_name', 'metric_value',
     'metric_comment', 'source_page']
]

# the declared grain must actually hold
duplicate_key_count = (
    new_gold_fact_enforcement_rows_df['enforcement_key'].duplicated().sum()
)
if duplicate_key_count:
    log("error", f"[{duplicate_key_count}] duplicate (fiscal_year, metric_name) pairs "
                 f"— the declared gold_fact_enforcement grain does not hold")

# stop rather than orphan a fact row against a missing dimension member
gold_dim_date_key_df = sql(conn, "SELECT date_key FROM gold_dim_date")
orphan_count = (~new_gold_fact_enforcement_rows_df['date_key'].isin(
    gold_dim_date_key_df['date_key'])).sum()
if orphan_count:
    log("error", f"[{orphan_count}] gold_fact_enforcement rows have a date_key with "
                 f"no matching gold_dim_date member")

# drop incoming rows that already exist in gold
gold_fact_enforcement_key_df = sql(
    conn, "SELECT enforcement_key FROM gold_fact_enforcement"
)
new_gold_fact_enforcement_rows_df = new_gold_fact_enforcement_rows_df[
    ~new_gold_fact_enforcement_rows_df['enforcement_key'].isin(
        gold_fact_enforcement_key_df['enforcement_key']
    )
]

if new_gold_fact_enforcement_rows_df.empty:
    log("warn", "no new rows to load into gold_fact_enforcement")
else:
    new_gold_fact_enforcement_rows_df.to_sql(
        "gold_fact_enforcement", conn, if_exists="append", index=False
    )
    log("info", f"loaded [{len(new_gold_fact_enforcement_rows_df)}] "
                f"into gold_fact_enforcement")


# gold_fact_assignment
new_gold_fact_assignment_rows_df = sql(
    conn,
    """SELECT
           agent_id,
           supervisor_id,
           region                AS region_name,
           role,
           department,
           unit,
           detention_center_name,
           assignment_start_date AS start_date,
           assignment_end_date   AS end_date
       FROM silver_operations""",
)

# resolve dimension keys with the same hashes used to build the dimensions
new_gold_fact_assignment_rows_df['org_unit_key'] = (
    new_gold_fact_assignment_rows_df.apply(
        lambda row: get_md5_hash(row['department'], row['unit']), axis=1
    )
)
new_gold_fact_assignment_rows_df['role_key'] = (
    new_gold_fact_assignment_rows_df.apply(
        lambda row: get_md5_hash(row['role']), axis=1
    )
)

facility_parts_df = new_gold_fact_assignment_rows_df['detention_center_name'].str.extract(
    r'^(?P<facility_name>.+?)\s*\((?P<state_code>[A-Z]{2})\)$'
)
new_gold_fact_assignment_rows_df['facility_key'] = facility_parts_df.apply(
    lambda row: get_md5_hash(row['facility_name'], row['state_code']), axis=1
)

new_gold_fact_assignment_rows_df['start_date_key'] = pd.to_datetime(
    new_gold_fact_assignment_rows_df['start_date']
).dt.strftime('%Y%m%d').astype('Int64')
new_gold_fact_assignment_rows_df['end_date_key'] = pd.to_datetime(
    new_gold_fact_assignment_rows_df['end_date']
).dt.strftime('%Y%m%d').astype('Int64')

assignment_start_dates = pd.to_datetime(new_gold_fact_assignment_rows_df['start_date'])
assignment_end_dates = pd.to_datetime(new_gold_fact_assignment_rows_df['end_date'])

2026-08-13T22:43:25 WARN: no new rows to load into gold_fact_enforcement


In [18]:
# print all tables and columns in database
pretty_print_db(conn)

Tables in the database:
╒════════════════════════════════╤═════════════════════╤══════════════════╕
│ Table Name                     │   Number of Columns │   Number of Rows │
╞════════════════════════════════╪═════════════════════╪══════════════════╡
│ bronze_ice_budget              │                   7 │              300 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_enforcement_metrics │                   7 │               20 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_enforcement_pdfs    │                   6 │              125 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_ice_operations          │                   9 │              400 │
├────────────────────────────────┼─────────────────────┼──────────────────┤
│ bronze_treasury_reconciliation │                  14 │              594 │
├────────────────────────────────┼─────────────────────┼────────

In [19]:
conn.close()